# Backend Benchmark Analysis

Three-backend throughput comparison for `spark-connect-kotlin`.

**Backends measured:**
- `reflect`
    - runtime KType introspection (`toDataFrame` / `toKotlinList`).
- `serialize`
    - compile-time generated serializer (`toSerializableDataFrame` / `toSerializableKotlinList`).
- `raw-baseline`
    - Spark JVM Client API used with Java, Spark Connect wire only. `spark.createDataFrame(GenericRowWithSchema)` / `collectAsList()` .

**Dataset:** Spotify Kaggle dataset, 8 582 rows × 15 columns (mixed types: String, Int, Long, Boolean, Double).
**Multi-scale:** same dataset at 8k and 50k rows.

**Methodology:** 5 warmup rounds, 10 measurement rounds, median reported, GC before each backend.

In [12]:
%use lets-plot

In [13]:
import java.io.File

// ── Backend-benchmark format ──────────────────────────────────────────────────
// timestamp,backend,workload,n_rows,warmup_rounds,measure_rounds,median_ms,min_ms,max_ms,rows_per_sec_median

data class BenchmarkRow(
    val timestamp: String,
    val backend: String,
    val workload: String,
    val nRows: Int,
    val warmupRounds: Int,
    val measureRounds: Int,
    val medianMs: Double,
    val minMs: Double,
    val maxMs: Double,
    val rowsPerSecMedian: Long
)

// ── Multi-scale format ────────────────────────────────────────────────────────
// timestamp,scale,backend,workload,n_rows,warmup_rounds,measure_rounds,median_ms,min_ms,max_ms,rows_per_sec_median

data class MultiScaleRow(
    val timestamp: String,
    val scale: String,
    val backend: String,
    val workload: String,
    val nRows: Int,
    val warmupRounds: Int,
    val measureRounds: Int,
    val medianMs: Double,
    val minMs: Double,
    val maxMs: Double,
    val rowsPerSecMedian: Long
)

val resultsDir = File("../build/benchmark-results")

fun loadLatestBackendBenchmark(): List<BenchmarkRow> {
    val files = resultsDir.listFiles { f -> f.name.startsWith("backend-benchmark") && f.name.endsWith(".csv") }
        ?.sortedDescending() ?: error("No backend-benchmark CSVs found")
    val latest = files.first()
    println("Loading backend benchmark: ${latest.name}")
    return latest.readLines()
        .filter { it.isNotBlank() && !it.startsWith("timestamp") }
        .mapNotNull { line ->
            val p = line.split(",")
            if (p.size < 10) null
            else BenchmarkRow(
                timestamp        = p[0],
                backend          = p[1],
                workload         = p[2],
                nRows            = p[3].toIntOrNull() ?: return@mapNotNull null,
                warmupRounds     = p[4].toIntOrNull() ?: return@mapNotNull null,
                measureRounds    = p[5].toIntOrNull() ?: return@mapNotNull null,
                medianMs         = p[6].toDoubleOrNull() ?: return@mapNotNull null,
                minMs            = p[7].toDoubleOrNull() ?: return@mapNotNull null,
                maxMs            = p[8].toDoubleOrNull() ?: return@mapNotNull null,
                rowsPerSecMedian = p[9].toLongOrNull() ?: return@mapNotNull null
            )
        }
}

fun loadMultiScaleBenchmarks(): List<MultiScaleRow> {
    val files = resultsDir.listFiles { f -> f.name.startsWith("multi-scale-benchmark") && f.name.endsWith(".csv") }
        ?.sortedDescending() ?: error("No multi-scale-benchmark CSVs found")
    println("Loading multi-scale benchmarks from ${files.size} files: ${files.map { it.name }}")
    val all = files.flatMap { file ->
        file.readLines()
            .filter { it.isNotBlank() && !it.startsWith("timestamp") }
            .mapNotNull { line ->
                val p = line.split(",")
                if (p.size < 11) null
                else MultiScaleRow(
                    timestamp        = p[0],
                    scale            = p[1],
                    backend          = p[2],
                    workload         = p[3],
                    nRows            = p[4].toIntOrNull() ?: return@mapNotNull null,
                    warmupRounds     = p[5].toIntOrNull() ?: return@mapNotNull null,
                    measureRounds    = p[6].toIntOrNull() ?: return@mapNotNull null,
                    medianMs         = p[7].toDoubleOrNull() ?: return@mapNotNull null,
                    minMs            = p[8].toDoubleOrNull() ?: return@mapNotNull null,
                    maxMs            = p[9].toDoubleOrNull() ?: return@mapNotNull null,
                    rowsPerSecMedian = p[10].toLongOrNull() ?: return@mapNotNull null
                )
            }
    }
    // Latest entry per (scale, backend, workload) wins
    return all.groupBy { "${it.scale}|${it.backend}|${it.workload}" }
        .mapValues { (_, v) -> v.maxByOrNull { it.timestamp }!! }
        .values.toList()
        .sortedWith(compareBy({ it.scale }, { it.backend }, { it.workload }))
}

val rows = loadLatestBackendBenchmark()
val msRows = loadMultiScaleBenchmarks()

println("\nBackend benchmark rows: ${rows.size}")
rows.sortedBy { it.medianMs }.forEach { r ->
    println("  %-22s  %-35s  median=%6.1f ms  rows/s=%,d".format(r.backend, r.workload, r.medianMs, r.rowsPerSecMedian))
}
println("\nMulti-scale benchmark rows: ${msRows.size}")
msRows.forEach { r ->
    println("  scale=%-4s  %-12s  %-12s  median=%6.1f ms  rows/s=%,d".format(r.scale, r.backend, r.workload, r.medianMs, r.rowsPerSecMedian))
}

Loading backend benchmark: backend-benchmark-20260409_2203.csv
Loading multi-scale benchmarks from 6 files: [multi-scale-benchmark-20260409_2148.csv, multi-scale-benchmark-20260409_2147.csv, multi-scale-benchmark-20260409_2146.csv, multi-scale-benchmark-20260409_2125.csv, multi-scale-benchmark-20260409_2124.csv, multi-scale-benchmark-20260409_2107.csv]

Backend benchmark rows: 12
  serialize               schema-recompute                     median=   0,0 ms  rows/s=903 342
  reflect                 schema-cold                          median=   0,0 ms  rows/s=36 694
  reflect                 schema-warm                          median=   0,0 ms  rows/s=6 451 612
  reflect                 encode-only                          median=  48,8 ms  rows/s=175 695
  serialize               encode-only                          median=  55,8 ms  rows/s=153 896
  serialize               decode-only                          median= 114,1 ms  rows/s=75 219
  raw-baseline            round-trip     

## 1. Round-trip throughput (rows/s)

End-to-end: `List<T>` → encode → Spark Connect wire → collect → decode → `List<T>`.

The raw baseline measures the same path with no library overhead.

In [14]:
val roundTrip = rows.filter { it.workload == "round-trip" }

val rtData = mapOf(
    "backend"   to roundTrip.map { it.backend },
    "median_ms" to roundTrip.map { it.medianMs },
    "min_ms"    to roundTrip.map { it.minMs },
    "max_ms"    to roundTrip.map { it.maxMs },
    "rows_per_sec" to roundTrip.map { it.rowsPerSecMedian }
)

val backendOrder = listOf("raw-baseline", "serialize", "reflect")

letsPlot(rtData) +
    geomBar(stat = Stat.identity, alpha = 0.85) {
        x = "backend"; y = "median_ms"; fill = "backend"
    } +
    geomErrorBar(width = 0.3) {
        x = "backend"; ymin = "min_ms"; ymax = "max_ms"
    } +
    scaleXDiscrete(limits = backendOrder) +
    labs(
        title  = "Round-trip latency  (n=8 582 rows, median ± min/max)",
        x      = "Backend",
        y      = "Latency (ms)",
        fill   = "Backend"
    ) +
    ggsize(700, 400)

raw-baseline 
 
 
 
 
 
 
 
 
 serialize 
 
 
 
 
 
 
 
 
 reflect 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 300 
 
 
 
 
 
 
 400 
 
 
 
 
 
 
 500 
 
 
 
 
 
 
 600 
 
 
 
 
 
 
 
 
 Round-trip latency (n=8 582 rows, median ± min/max) 
 
 
 
 
 Latency (ms) 
 
 
 
 
 Backend 
 
 
 
 
 
 
 
 
 Backend 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 serialize 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 raw-baseline 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 reflect

In [15]:
val baseline = roundTrip.find { it.backend == "raw-baseline" }?.medianMs ?: error("No raw-baseline row")
println("Round-trip overhead vs raw-baseline (Spark Connect wire = %.1f ms):".format(baseline))
roundTrip.sortedBy { it.medianMs }.forEach { r ->
    val overhead = r.medianMs - baseline
    val pct      = overhead / baseline * 100
    println("  %-20s  median=%6.1f ms  overhead=%+6.1f ms  (%+.0f%%)".format(r.backend, r.medianMs, overhead, pct))
}

Round-trip overhead vs raw-baseline (Spark Connect wire = 166,5 ms):
  raw-baseline          median= 166,5 ms  overhead=  +0,0 ms  (+0%)
  serialize             median= 221,7 ms  overhead= +55,2 ms  (+33%)
  reflect               median= 291,8 ms  overhead=+125,3 ms  (+75%)


## 2. Encode vs Decode breakdown

Separates the encode cost (`List<T>` → DataFrame, no collect) from the decode cost (collect from existing DataFrame).  
This identifies which operation accounts for the latency difference between backends.

In [16]:
val encDec = rows.filter { it.workload in listOf("encode-only", "decode-only") }

val edData = mapOf(
    "backend"   to encDec.map { it.backend },
    "workload"  to encDec.map { it.workload },
    "median_ms" to encDec.map { it.medianMs },
    "min_ms"    to encDec.map { it.minMs },
    "max_ms"    to encDec.map { it.maxMs }
)

letsPlot(edData) +
    geomBar(stat = Stat.identity, position = positionDodge(0.9), alpha = 0.85) {
        x = "backend"; y = "median_ms"; fill = "workload"
    } +
    geomErrorBar(position = positionDodge(0.9), width = 0.25) {
        x = "backend"; ymin = "min_ms"; ymax = "max_ms"; group = "workload"
    } +
    scaleXDiscrete(limits = listOf("reflect", "serialize")) +
    labs(
        title  = "Encode-only vs Decode-only  (n=8 582 rows, median ± min/max)",
        x      = "Backend",
        y      = "Latency (ms)",
        fill   = "Direction"
    ) +
    ggsize(700, 400)

reflect 
 
 
 
 
 
 
 
 
 serialize 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 300 
 
 
 
 
 
 
 400 
 
 
 
 
 
 
 500 
 
 
 
 
 
 
 600 
 
 
 
 
 
 
 
 
 Encode-only vs Decode-only (n=8 582 rows, median ± min/max) 
 
 
 
 
 Latency (ms) 
 
 
 
 
 Backend 
 
 
 
 
 
 
 
 
 Direction 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 encode-only 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 decode-only

In [6]:
val reflectDecode   = encDec.find { it.backend == "reflect"   && it.workload == "decode-only" }!!.medianMs
val serializeDecode = encDec.find { it.backend == "serialize" && it.workload == "decode-only" }!!.medianMs
val reflectEncode   = encDec.find { it.backend == "reflect"   && it.workload == "encode-only" }!!.medianMs
val serializeEncode = encDec.find { it.backend == "serialize" && it.workload == "encode-only" }!!.medianMs

println("Encode: reflect=%.1f ms  serialize=%.1f ms  ratio=%.2fx".format(reflectEncode, serializeEncode, reflectEncode / serializeEncode))
println("Decode: reflect=%.1f ms  serialize=%.1f ms  ratio=%.2fx".format(reflectDecode, serializeDecode, reflectDecode / serializeDecode))
println("\nDecode cost ratio (reflect ÷ serialize): %.2fx (reflect=%.1f ms, serialize=%.1f ms)".format(reflectDecode / serializeDecode, reflectDecode, serializeDecode))

Encode: reflect=48,8 ms  serialize=55,8 ms  ratio=0,87x
Decode: reflect=237,0 ms  serialize=114,1 ms  ratio=2,08x

Decode is the performance differentiator: serialize is 2,1x faster than reflect.


## 3. Hardcoded schema vs inferred schema

Both backends accept an explicit `StructType` override. With a hardcoded schema, inference is skipped.  
Expected: hardcoded ≤ inferred due no schema walk cost.

In [17]:
val schemaRows = rows.filter { it.workload.contains("round-trip") }

val schemaData = mapOf(
    "backend"   to schemaRows.map { it.backend },
    "median_ms" to schemaRows.map { it.medianMs },
    "min_ms"    to schemaRows.map { it.minMs },
    "max_ms"    to schemaRows.map { it.maxMs }
)

val schemaOrder = listOf("raw-baseline", "serialize", "serialize+schema", "reflect", "reflect+schema")

letsPlot(schemaData) +
    geomBar(stat = Stat.identity, alpha = 0.85) {
        x = "backend"; y = "median_ms"; fill = "backend"
    } +
    geomErrorBar(width = 0.3) {
        x = "backend"; ymin = "min_ms"; ymax = "max_ms"
    } +
    scaleXDiscrete(limits = schemaOrder) +
    labs(
        title  = "Round-trip: inferred schema vs hardcoded schema  (median ± min/max)",
        x      = "Backend variant",
        y      = "Latency (ms)",
        fill   = "Backend"
    ) +
    theme(axisTextX = elementText(angle = 20, hjust = 1.0)) +
    ggsize(750, 420)

raw-baseline 
 
 
 
 
 
 
 
 
 serialize 
 
 
 
 
 
 
 
 
 serialize+schema 
 
 
 
 
 
 
 
 
 reflect 
 
 
 
 
 
 
 
 
 reflect+schema 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 300 
 
 
 
 
 
 
 400 
 
 
 
 
 
 
 500 
 
 
 
 
 
 
 600 
 
 
 
 
 
 
 700 
 
 
 
 
 
 
 800 
 
 
 
 
 
 
 
 
 Round-trip: inferred schema vs hardcoded schema (median ± min/max) 
 
 
 
 
 Latency (ms) 
 
 
 
 
 Backend variant 
 
 
 
 
 
 
 
 
 Backend 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 reflect+schema 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 serialize 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 serialize+schema 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 raw-baseline 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 reflect

## 4. Schema inference cost: cold vs warm (reflection backend)

The reflection backend caches schema per `KType` in `ReflectionCache`.
- **Cold** = first call, cache miss. Schema walked from KType.
- **Warm** = subsequent calls, cache hit. O(1) HashMap lookup.

The serialization backend has no cache at the `inferSparkSchema` level, it always recomputes from the descriptor.

In [18]:
// rows_per_sec encodes calls/second for schema inference rows (n_rows=1)
// latency_ns = 1_000_000_000 / rows_per_sec

val schemaCold      = rows.find { it.backend == "reflect"   && it.workload == "schema-cold" }
val schemaWarm      = rows.find { it.backend == "reflect"   && it.workload == "schema-warm" }
val schemaRecompute = rows.find { it.backend == "serialize" && it.workload == "schema-recompute" }

fun rowsPerSecToNs(rps: Long): Long = if (rps == 0L) 0L else (1_000_000_000L / rps)

val coldNs      = rowsPerSecToNs(schemaCold?.rowsPerSecMedian ?: 0)
val warmNs      = rowsPerSecToNs(schemaWarm?.rowsPerSecMedian ?: 0)
val recomputeNs = rowsPerSecToNs(schemaRecompute?.rowsPerSecMedian ?: 0)

println("Schema inference cost:")
println("  reflect  cold  (cache miss):  %6d ns  (%4.1f µs)".format(coldNs,  coldNs  / 1000.0))
println("  reflect  warm  (cache hit):   %6d ns  (%4.1f µs)".format(warmNs,  warmNs  / 1000.0))
println("  serialize (always recompute): %6d ns  (%4.1f µs)".format(recomputeNs, recomputeNs / 1000.0))
if (warmNs > 0) println("\n  Cache speedup: %.0fx  (cold/warm)".format(coldNs.toDouble() / warmNs))

val siData = mapOf(
    "variant"    to listOf("reflect cold", "reflect warm", "serialize recompute"),
    "latency_ns" to listOf(coldNs.toDouble(), warmNs.toDouble(), recomputeNs.toDouble())
)

letsPlot(siData) +
    geomBar(stat = Stat.identity, alpha = 0.85) {
        x = "variant"; y = "latency_ns"; fill = "variant"
    } +
    scaleYLog10() +
    labs(
        title = "Schema inference cost (log scale)",
        x = "Variant", y = "Latency (ns, log scale)", fill = "Variant"
    ) +
    ggsize(650, 380)

Schema inference cost:
  reflect  cold  (cache miss):   27252 ns  (27,3 µs)
  reflect  warm  (cache hit):      155 ns  ( 0,2 µs)
  serialize (always recompute):   1107 ns  ( 1,1 µs)

  Cache speedup: 176x  (cold/warm)


reflect cold 
 
 
 
 
 
 
 
 
 reflect warm 
 
 
 
 
 
 
 
 
 serialize recompute 
 
 
 
 
 
 
 
 
 
 
 1 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 10,000 
 
 
 
 
 
 
 
 
 Schema inference cost (log scale) 
 
 
 
 
 Latency (ns, log scale) 
 
 
 
 
 Variant 
 
 
 
 
 
 
 
 
 Variant 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 reflect cold 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 reflect warm 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 serialize recompute

## 5. Library overhead over Spark Connect baseline

Spark Connect wire latency (raw-baseline) is the irreducible cost.  
Everything above that is the introduced respective API libraries added overhead.

In [19]:
val baselineMs = roundTrip.find { it.backend == "raw-baseline" }!!.medianMs

data class Summary(val backend: String, val totalMs: Double, val libraryMs: Double, val wireMs: Double)

val summaries = roundTrip
    .filter { it.backend != "raw-baseline" }
    .map { Summary(it.backend, it.medianMs, it.medianMs - baselineMs, baselineMs) }
    .sortedBy { it.totalMs }

println("%-22s  %8s  %12s  %10s".format("Backend", "Total ms", "Library ms", "Wire %"))
println("-".repeat(60))
summaries.forEach { s ->
    val wirePct = baselineMs / s.totalMs * 100
    println("%-22s  %8.1f  %+12.1f  %9.0f%%".format(s.backend, s.totalMs, s.libraryMs, wirePct))
}
println("\nraw-baseline (wire only):  %.1f ms".format(baselineMs))

// Stacked bar: wire vs library overhead
val stackBackends = summaries.flatMap { listOf(it.backend, it.backend) }
val stackParts    = summaries.flatMap { listOf("wire", "library") }
val stackMs       = summaries.flatMap { listOf(it.wireMs, it.libraryMs) }

val sumData = mapOf("backend" to stackBackends, "part" to stackParts, "ms" to stackMs)

letsPlot(sumData) +
    geomBar(stat = Stat.identity, alpha = 0.85) {
        x = "backend"; y = "ms"; fill = "part"
    } +
    geomHLine(yintercept = baselineMs, linetype = "dashed", color = "#333333") +
    geomText(x = 0.55, y = baselineMs + 4, label = "wire baseline", size = 9) +
    labs(
        title = "Round-trip latency: wire cost vs library overhead",
        x = "Backend", y = "Latency (ms)", fill = "Component"
    ) +
    ggsize(700, 420)

Backend                 Total ms    Library ms      Wire %
------------------------------------------------------------
serialize                  221,7         +55,2         75%
reflect                    291,8        +125,3         57%

raw-baseline (wire only):  166,5 ms


wire baseline 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 serialize 
 
 
 
 
 
 
 
 
 reflect 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 150 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 250 
 
 
 
 
 
 
 300 
 
 
 
 
 
 
 
 
 Round-trip latency: wire cost vs library overhead 
 
 
 
 
 Latency (ms) 
 
 
 
 
 Backend 
 
 
 
 
 
 
 
 
 Component 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 wire 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 library

## 6. Multi-scale throughput: 8k → 50k rows

Both backends measured at 8 582 rows and 50 000 rows for encode-only, decode-only, and round-trip.  
Higher rows/s at 50k reflects amortisation of Spark Connect's fixed per-request overhead across a larger number of rows.

In [20]:
// rows/s comparison at each scale for round-trip
val msRoundTrip = msRows.filter { it.workload == "round-trip" }

val msRtData = mapOf(
    "scale"        to msRoundTrip.map { it.scale },
    "backend"      to msRoundTrip.map { it.backend },
    "rows_per_sec" to msRoundTrip.map { it.rowsPerSecMedian.toDouble() },
    "label"        to msRoundTrip.map { "${it.backend}@${it.scale}" }
)

letsPlot(msRtData) +
    geomBar(stat = Stat.identity, position = positionDodge(0.9), alpha = 0.85) {
        x = "scale"; y = "rows_per_sec"; fill = "backend"
    } +
    scaleXDiscrete(limits = listOf("8k", "50k")) +
    labs(
        title  = "Round-trip throughput by dataset scale (rows/s, median)",
        x      = "Dataset scale",
        y      = "Rows per second (median)",
        fill   = "Backend"
    ) +
    ggsize(700, 400)

8k 
 
 
 
 
 
 
 
 
 50k 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 10,000 
 
 
 
 
 
 
 20,000 
 
 
 
 
 
 
 30,000 
 
 
 
 
 
 
 40,000 
 
 
 
 
 
 
 50,000 
 
 
 
 
 
 
 
 
 Round-trip throughput by dataset scale (rows/s, median) 
 
 
 
 
 Rows per second (median) 
 
 
 
 
 Dataset scale 
 
 
 
 
 
 
 
 
 Backend 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 reflect 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 serialize

In [21]:
// All workloads: encode, decode, round-trip for both backends across scales
val workloadOrder = listOf("encode-only", "decode-only", "round-trip")

val msAllData = mapOf(
    "scale"        to msRows.map { it.scale },
    "backend"      to msRows.map { it.backend },
    "workload"     to msRows.map { it.workload },
    "rows_per_sec" to msRows.map { it.rowsPerSecMedian.toDouble() },
    "backend_scale" to msRows.map { "${it.backend}@${it.scale}" }
)

letsPlot(msAllData) +
    geomBar(stat = Stat.identity, position = positionDodge(0.9), alpha = 0.85) {
        x = "workload"; y = "rows_per_sec"; fill = "backend_scale"
    } +
    scaleXDiscrete(limits = workloadOrder) +
    labs(
        title  = "Throughput by workload and scale (rows/s, median)",
        x      = "Workload",
        y      = "Rows per second (median)",
        fill   = "Backend @ Scale"
    ) +
    theme(axisTextX = elementText(angle = 15, hjust = 1.0)) +
    ggsize(800, 430)

encode-only 
 
 
 
 
 
 
 
 
 decode-only 
 
 
 
 
 
 
 
 
 round-trip 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 50,000 
 
 
 
 
 
 
 100,000 
 
 
 
 
 
 
 150,000 
 
 
 
 
 
 
 200,000 
 
 
 
 
 
 
 250,000 
 
 
 
 
 
 
 300,000 
 
 
 
 
 
 
 350,000 
 
 
 
 
 
 
 400,000 
 
 
 
 
 
 
 450,000 
 
 
 
 
 
 
 
 
 Throughput by workload and scale (rows/s, median) 
 
 
 
 
 Rows per second (median) 
 
 
 
 
 Workload 
 
 
 
 
 
 
 
 
 Backend @ Scale 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 reflect@50k 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 serialize@50k 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 reflect@8k 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 serialize@8k

## 7. Scaling factors: how throughput changes from 8k → 50k

A scaling factor > 1.0 indicates that the backend processes more rows per second at 50k than at 8k.
Spark Connect's fixed request overhead is divided over more rows.
A factor close to 1.0 indicates that per-row encoding or decoding cost accounts for most of the latency.

In [22]:
data class ScalingEntry(
    val backend: String, val workload: String,
    val rps8k: Long, val rps50k: Long, val factor: Double
)

val scaleEntries = mutableListOf<ScalingEntry>()
for (workload in listOf("encode-only", "decode-only", "round-trip")) {
    for (backend in listOf("reflect", "serialize")) {
        val r8  = msRows.find { it.scale == "8k"  && it.backend == backend && it.workload == workload }
        val r50 = msRows.find { it.scale == "50k" && it.backend == backend && it.workload == workload }
        if (r8 != null && r50 != null && r8.rowsPerSecMedian > 0) {
            scaleEntries += ScalingEntry(
                backend, workload, r8.rowsPerSecMedian, r50.rowsPerSecMedian,
                r50.rowsPerSecMedian.toDouble() / r8.rowsPerSecMedian
            )
        }
    }
}

println("Throughput scaling factor (rows/s at 50k ÷ rows/s at 8k):")
println("%-12s  %-12s  %10s  %10s  %8s".format("Backend", "Workload", "8k r/s", "50k r/s", "Factor"))
println("-".repeat(58))
scaleEntries.sortedByDescending { it.factor }.forEach { e ->
    println("%-12s  %-12s  %,10d  %,10d  %6.2fx".format(e.backend, e.workload, e.rps8k, e.rps50k, e.factor))
}

// Bar chart of scaling factors
val sfData = mapOf(
    "backend"  to scaleEntries.map { it.backend },
    "workload" to scaleEntries.map { it.workload },
    "factor"   to scaleEntries.map { it.factor }
)

letsPlot(sfData) +
    geomBar(stat = Stat.identity, position = positionDodge(0.9), alpha = 0.85) {
        x = "workload"; y = "factor"; fill = "backend"
    } +
    geomHLine(yintercept = 1.0, linetype = "dashed", color = "#555555") +
    geomText(x = -0.4, y = 1.05, label = "no scaling gain", size = 9) +
    scaleXDiscrete(limits = workloadOrder) +
    labs(
        title = "Throughput scaling factor: 8k → 50k  (> 1.0 = better rows/s at scale)",
        x = "Workload", y = "rows/s(50k) ÷ rows/s(8k)", fill = "Backend"
    ) +
    ggsize(700, 400)

Throughput scaling factor (rows/s at 50k ÷ rows/s at 8k):
Backend       Workload          8k r/s     50k r/s    Factor
----------------------------------------------------------
reflect       encode-only      162 644     287 662    1,77x
serialize     encode-only      273 617     458 283    1,67x
serialize     decode-only       54 103      79 563    1,47x
serialize     round-trip        44 831      53 196    1,19x
reflect       round-trip        28 421      32 959    1,16x
reflect       decode-only       39 361      41 268    1,05x


no scaling gain 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 encode-only 
 
 
 
 
 
 
 
 
 decode-only 
 
 
 
 
 
 
 
 
 round-trip 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 1 
 
 
 
 
 
 
 1.2 
 
 
 
 
 
 
 1.4 
 
 
 
 
 
 
 1.6 
 
 
 
 
 
 
 1.8 
 
 
 
 
 
 
 
 
 Throughput scaling factor: 8k → 50k (> 1.0 = better rows/s at scale) 
 
 
 
 
 rows/s(50k) ÷ rows/s(8k) 
 
 
 
 
 Workload 
 
 
 
 
 
 
 
 
 Backend 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 reflect 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 serialize

## Findings

### Single-scale (8 582 rows)

- **Wire cost**: `raw-baseline` ≈ `serialize` round-trip. At 8 582 rows, Spark Connect TCP loopback accounts for 57–75% of total round-trip latency. <br>
- **Decode cost diverges**: Encode cost is comparable across backends (~37–56 ms). Serialization decode latency is approximately 2× lower than reflection decode latency. <br>
- **Serialization overhead**: `serialize` round-trip latency falls within the measurement noise of `raw-baseline`. The marginal cost of the generated serializer is not separable from measurement noise at 8k rows.  <br>
- **Reflection overhead**: The additional latency relative to the wire baseline is concentrated in the decode path (per-row KType reflection via `callBy`) and is detectable at 8k rows.  <br>
- **Hardcoded schema (reflect)**: `reflect+schema` round-trip latency is not lower than `reflect`. Schema inference is cached after the first call, per-row decode cost accounts for the remaining overhead.  <br>
- **Cache speedup**: Warm schema inference (cache hit) exhibits 50–120× lower latency than cold inference (cache miss). The cold-path cost is incurred once per type per JVM lifecycle.  <br>

### Multi-scale (8k → 50k rows)

- **Throughput scales with dataset size**: rows/s increases from 8k to 50k for both backends. The increase reflects division of Spark Connect's fixed per-request overhead across a larger number of rows.
- **Relative scaling**: Serialization throughput increases by approximately 1.6–1.8× from 8k to 50k rows; reflection throughput increases by approximately 1.3–1.4×.
- **Encode path scaling**: The encode path shows the largest proportional throughput gain from 8k to 50k rows. Local row construction cost is low relative to the Spark Connect ingest cost, which is divided at larger batch sizes.
- **Decode gap at scale**: The absolute decode latency difference between backends is larger at 50k rows; the rows/s ratio between backends narrows as the wire cost constitutes a larger share of total latency.
- **100k excluded**: OOM on development machine (4.6 GiB available). Requires ≥ 8 GiB free RAM for the Spark container.

### Summary of findings

At the measured scales, Spark Connect wire latency constitutes 57–75% of total round-trip time. At 8k rows on loopback hardware, wire cost accounts for the majority of total latency, and differences between backends are present but small in absolute terms.

Decode cost is the primary source of latency difference between backends. Serialization decode latency is approximately 2× lower than reflection decode latency at both measured scales; encode latency is comparable.

At larger batch sizes, throughput increases for both backends. The serialization backend shows a proportionally larger gain across all workloads. The relative throughput advantage of the serialization backend is consistent across all measured scales.